In [1]:
import onnxruntime as rt
rt.get_device()
import json
# train_path = "/home/andrew/cafa5_team/data/"
# with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
#     go_terms = json.load(f)
with open('/home/andrew/GO_interp/data/proteinf_go_terms.json', 'r') as f:
    go_terms_pi = json.load(f)
sess = rt.InferenceSession("/home/andrew/GO_interp/data/proteinf_model.onnx")
import pickle
AMINO_ACID_VOCABULARY = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
from collections import defaultdict
RESIDUE_TO_INT = defaultdict(lambda : 0, {aa: idx for idx, aa in enumerate(AMINO_ACID_VOCABULARY)})
import numpy as np
oh_mat = np.eye(len(RESIDUE_TO_INT), dtype=np.float32)
def get_onehot(seq_ind):
    seq_oh = oh_mat[seq_ind]
    return seq_oh
def get_seq_logit(seq):
    seq_len = len(seq)
    batch_seq = np.array([RESIDUE_TO_INT[c] for c in seq]).reshape((1, -1))
    batch_len = np.full((1,), seq_len, dtype=np.int32)
    batch_seq_oh = get_onehot(batch_seq)
    logits = sess.run(output_names=['output'], input_feed={'sequence_length': batch_len, 'sequence': batch_seq_oh})
    return logits

In [2]:
get_seq_logit("MSSSSTMSSSSTMSSSSTMSSSSTMSSSSTMSSSST")[0].shape

(1, 32102)

In [3]:
import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "rb") as f:
    train_dataset = pickle.load(f)
with open(f"{data_path}/val_dataset.pkl", "rb") as f:
    val_dataset = pickle.load(f)
# from go_ml.data_utils import *
# val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn": prot_func_collate}

In [ ]:
labels = val_dataset.labels

In [4]:
from tqdm.notebook import tqdm
val_logits = []
for seq in tqdm(val_dataset.sequences):
    logits = get_seq_logit(seq)
    val_logits.append(logits[0])

  0%|          | 0/31131 [00:00<?, ?it/s]

In [ ]:
concat_logits = np.concatenate(val_logits, axis=0)
print(concat_logits.shape, labels.shape)
concat_logits = np.concatenate([concat_logits, np.zeros_like(concat_logits[:, :1])], axis=1)
print(concat_logits.shape)

(31131, 32102) (31131, 29185)


In [16]:
import pickle
with open("pi_val_preds.pkl", "wb") as f:
    pickle.dump(concat_logits, f)

In [6]:
import json
with open('/home/andrew/GO_interp/data/proteinf_go_terms.json', 'r') as f:
    go_terms_pi = json.load(f)

train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)

In [10]:
go_term_ind = {go_term: idx for idx, go_term in enumerate(go_terms)}
pi_term_ind = {go_term: idx for idx, go_term in enumerate(go_terms_pi)}
pi_ref_ind = []
for go_term in go_terms:
    if go_term in pi_term_ind:
        pi_ref_ind.append(pi_term_ind[go_term])
    else:
        pi_ref_ind.append(len(pi_term_ind))
pi_ref_ind = np.array(pi_ref_ind, dtype=np.int32)

In [17]:
remapped_logits = concat_logits[:, pi_ref_ind]
print(remapped_logits.shape)

(31131, 29185)


In [28]:
remapped_logits = remapped_logits[:, pi_ref_ind < len(go_terms_pi)]

In [32]:
labels = labels.toarray()

In [33]:
labels = labels[:, pi_ref_ind < len(go_terms_pi)]

In [34]:
from sklearn.metrics import f1_score
f1 = f1_score(labels, remapped_logits > 0.5, average='micro')
print(f"F1 Score: {f1:.4f}")

F1 Score: 0.6689


In [ ]:
import onnx
import torch
from onnx2torch import convert

onnx_model_path = "/home/andrew/GO_interp/data/proteinf_model.onnx"
# You can pass the path to the onnx model to convert it or...
torch_model_1 = convert(onnx_model_path)

# Or you can load a regular onnx model and pass it to the converter
onnx_model = onnx.load(onnx_model_path)
torch_model_2 = convert(onnx_model)

NotImplementedError: "SAME_UPPER" auto_pad is not implemented